# NB-07 — RQ4: Does the Attack Survive a Deployment Pipeline?

**One knob: `RUNG`.** Run it four times, once per rung. Everything else is fixed.

---

## The question, and its honest scope

Every attack in this thesis so far is a **digital file** handed straight to the model. A real
deployment chain does not do that. It compresses, resamples, and adds noise. RQ4 asks whether a
perturbation that works on the raw waveform still works after that chain — and, because that is
the point of the thesis, **whether survival depends on how much data the victim was trained on.**

**In scope (this notebook).** Codec re-encode (MP3, AAC, Opus at deployment bitrates),
sample-rate round-trip through 8 kHz, µ-law companding, and additive noise at fixed SNR. All
digital, all reproducible on Kaggle, no hardware.

**Out of scope, and we say so in the paper rather than leaving it pending.**

| Excluded | Why |
|---|---|
| **Over-the-air playback** | Needs a speaker, a microphone and a controlled room. No hardware budget. This is the single biggest remaining gap in the threat model and the paper must keep saying so. |
| **Psychoacoustic masking** | Mature line already (Qin~et~al., Schönherr~et~al.). We would be reimplementing, not contributing. Our novelty is the data-volume axis, not the masking. |
| **VAD gating** | Depends on a specific vendor's front end; not generalisable. |

So this notebook does not close RQ4. It converts RQ4 from *"not implemented"* into *"measured
for the digital chain; over-the-air still open"*, which is a defensible thing to publish.

## Why it must re-run the attack

NB-05 saved only per-utterance **metrics**, never the adversarial audio or the perturbation. So
the waveforms do not exist and cannot be transformed. This notebook regenerates them at a fixed
budget, transforms them, and **writes the WAVs to its output** so nobody pays this cost twice.

## What it measures

For each utterance where the attack succeeds on the raw waveform, it applies each transform and
re-decodes with the same model. Three outcomes per transform:

| Outcome | Meaning |
|---|---|
| **survives** | still an exact match to the attacker's target |
| **broken** | no longer the target, and not the true transcript either — the transform destroyed the attack but left the audio unusable |
| **reverted** | the model now returns the *true* transcript — the transform fully repaired the input |

`survival_rate` is the headline: successful attacks that remain successful. The `reverted` count
matters separately, because a cheap transform that reverts attacks is a **deployable defence**,
which is the same argument RQ5 makes for the language model.

---

## Kaggle settings

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** (ffmpeg codec libraries) |
| Persistence | Variables and Files |

### Inputs — three, plus the rung you are attacking

| # | Dataset |
|---|---|
| 1 | `bengaliai-speech` |
| 2 | `bn-asr-splits` |
| 3 | `ckpt-xlsr-{RUNG}h` for the rung set below |

Do **not** attach `bn-attack-targets`. This uses the 25-utterance RQ1 set so the survival numbers
sit on the same utterances as the ladder.

### The four runs

| # | `RUNG` | Publishes as |
|---|---|---|
| 1 | `1` | `pipeline-survival-rung1` |
| 2 | `5` | `pipeline-survival-rung5` |
| 3 | `25` | `pipeline-survival-rung25` |
| 4 | `100` | `pipeline-survival-rung100` |

~25 min each, so **~1.7 GPU-h for the ladder**. Two sessions in parallel is fine.

### Read these lines before trusting a run

1. `eval set [t5_frozen]: .../attack_eval_25.csv` and `25 utterances`
2. `PREFLIGHT PASSED`
3. `ffmpeg OK` and a transform list with **no** `SKIPPED` entries
4. `raw attack success: N/25` — if this is 0 there is nothing to transform and survival is undefined


## 1 · Configuration — one knob

In [ ]:
# ============================================================================
#  THE ONLY SETTING YOU CHANGE
# ============================================================================
RUNG = 1                 # <<<< 1, 5, 25, or 100

# ============================================================================
#  LOCKED. Survival must sit on the same utterances and the same budget as the
#  RQ1 ladder, or it cannot be read against exposure.
# ============================================================================
EDIT_TYPE    = "T5"
CHECKPOINT   = "final"       # rung finals only: the ladder's own conditions
EVAL_SET     = "t5_frozen"   # attack_eval_25.csv
FIXED_EPS    = 2e-3          # same ceiling as every trajectory condition
N_EVAL       = 25
CKPT_DATASET = None          # derives ckpt-xlsr-{RUNG}h

assert RUNG in (1, 5, 25, 100), f"RUNG={RUNG} is not a ladder rung"
_LOCKED = dict(EDIT_TYPE="T5", CHECKPOINT="final", EVAL_SET="t5_frozen",
               FIXED_EPS=2e-3, N_EVAL=25, CKPT_DATASET=None)
_actual = dict(EDIT_TYPE=EDIT_TYPE, CHECKPOINT=CHECKPOINT, EVAL_SET=EVAL_SET,
               FIXED_EPS=FIXED_EPS, N_EVAL=N_EVAL, CKPT_DATASET=CKPT_DATASET)
assert _actual == _LOCKED, f"locked setting edited\n  want {_LOCKED}\n  got  {_actual}"

# ---------------------------------------------------------------------------
#  The transform chain. Each entry is (name, kind, parameter).
#  Bitrates are what deployed voice pipelines actually use, not worst cases:
#  telephony Opus sits near 24 kbps, consumer voice messaging near 64 kbps MP3.
# ---------------------------------------------------------------------------
TRANSFORMS = [
    ("mp3_128",   "codec",    ("mp3",  "128k")),
    ("mp3_64",    "codec",    ("mp3",  "64k")),
    ("aac_64",    "codec",    ("aac",  "64k")),
    ("opus_24",   "codec",    ("opus", "24k")),
    ("resample8k","resample", 8000),
    ("mulaw",     "mulaw",    None),
    ("noise_30db","noise",    30.0),
    ("noise_20db","noise",    20.0),
]
SAVE_WAVS = True     # write raw + adversarial WAVs so this cost is paid once
print(f"NB-07  pipeline survival  |  RUNG = {RUNG}h")
print(f"  {len(TRANSFORMS)} transforms, fixed budget {FIXED_EPS:.1e}")
print("locked config OK\n")

# --- unchanged operational settings ---------------------------------------
BATCH_SIZE  = 8          # per-batch utterances; drop to 4 if you hit OOM
USE_AMP     = True       # settled by NB-04b run 2: AMP reproduced fp32 eps50 exactly
SESSION_HOURS = 8.0      # Kaggle kills a committed GPU session at 12 h
VALIDATE_BATCHING = False

EPS_LO, EPS_HI = (1e-5, 2e-3) if EDIT_TYPE == "T1" else (1e-4, 2e-3)
BISECT_ROUNDS  = 6 if EDIT_TYPE == "T1" else 5
if FIXED_EPS is not None:
    EPS_HI = float(FIXED_EPS)
    EPS_LO = min(EPS_LO, EPS_HI)
    BISECT_ROUNDS = 0
PROBE_STEPS    = 400     # steps per bisection probe (matches NB-04b)
CONFIRM_STEPS  = 800     # steps for the final confirmation at the chosen eps
STALL_PATIENCE = 100     # abort a probe after this many steps with no loss improvement
CHECK_EVERY    = 50
LR             = 1e-4

# Costs roughly THREE extra eps50 searches (~1-2 GPU-h) because it runs the
# same two utterances batched and then one at a time. Run it ONCE, for
# (RUNG=1, EDIT_TYPE="T5"), record the agreement in RUNLOG.md, then set it
# False for the remaining seven runs. It is a one-off correctness check on the
# batching, not something each run needs to repeat.
VALIDATE_BATCHING = False   # <<<< set True for ONE run only, then back to False.
                            # NOTE: RUNLOG has no row recording the batching
                            # agreement figure. It was almost certainly measured
                            # on the first run (RUNG=1, T5 -- the old defaults),
                            # but check that run's log and add the number to
                            # RUNLOG before the thesis cites batched eps50.

## 2 · Boilerplate

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # attacks optimise the INPUT: never DataParallel
import sys, json, time, random, re, glob, unicodedata
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP, DEVICE = "/kaggle/working", "/kaggle/temp", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"

PROV = {"notebook": "NB-07-pipeline-survival", "rung": RUNG, "edit_type": EDIT_TYPE,
        "n_eval": N_EVAL, "use_amp": USE_AMP, "seed": SEED,
        "bracket": [EPS_LO, EPS_HI], "bisect_rounds": BISECT_ROUNDS,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# See NB-01 for the full explanation: pinning an old `accelerate`/`datasets`
# silently downgrades numpy mid-session and breaks pandas much later with
# "No module named 'numpy.rec'". Pin numpy to what is already loaded.
# `datasets` is not imported by this notebook, so it is not installed.
import numpy, subprocess, sys
NP_BEFORE = numpy.__version__

!pip install -q transformers==4.44.2 librosa soundfile jiwer "numpy=={NP_BEFORE}" 2>&1 | tail -2

NP_DISK = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
assert NP_BEFORE == NP_DISK, f"numpy moved {NP_BEFORE} -> {NP_DISK}; relax the pin that forced it."
print(f"numpy {NP_BEFORE} stable")

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa
print("transformers", transformers.__version__)
PROV["transformers"] = transformers.__version__
PROV["numpy"] = NP_BEFORE


## 3 · Load the ladder checkpoint and the frozen attack set

In [ ]:
# ---------------------------------------------------------------------------
# Kaggle does not use one mount layout. Depending on the kernel you get
#     /kaggle/input/<name>                (older kernels)
#     /kaggle/input/datasets/<name>       (your own datasets, current)
#     /kaggle/input/competitions/<name>   (competition data)
# A flat glob only sees the wrapper dirs -- NB-01 failed with exactly this,
# reporting: Visible inputs: ['competitions', 'datasets']
# ---------------------------------------------------------------------------
inputs = sorted({p for pat in ("/kaggle/input/*", "/kaggle/input/*/*",
                               "/kaggle/input/*/*/*")
                 for p in glob.glob(pat) if os.path.isdir(p)})

# Match the rung EXACTLY. The old test was `f"{RUNG}h" in p`, and "5h" is a
# substring of "ckpt-xlsr-25h" -- with both datasets attached, RUNG=5 could
# silently attack the 25 h model and corrupt the headline figure.
_DS = CKPT_DATASET or f"ckpt-xlsr-{RUNG}h"
cands = [p for p in inputs if re.search(rf"{re.escape(_DS)}$", os.path.basename(p))]
assert len(cands) == 1, (
    f"Need exactly one input matching '{_DS}', found {len(cands)}: "
    f"{[os.path.basename(c) for c in cands]}\n"
    f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
CKPT = cands[0]
if CHECKPOINT == "final":
    MODEL_DIR = CKPT if os.path.exists(f"{CKPT}/config.json") else f"{CKPT}/final"
else:
    MODEL_DIR = f"{CKPT}/milestones/step-{CHECKPOINT.split('-')[1]}"
    assert os.path.isdir(MODEL_DIR), (
        f"{MODEL_DIR} missing. Milestones live at steps 1000/2000/3500. They exist "
        f"for rungs >= 25h in ckpt-xlsr-{{25,100}}h, and for the 1h rung ONLY in "
        f"ckpt-xlsr-1h-traj (the 2026-09-26 TRAJ_RUN retrain) -- the original "
        f"ckpt-xlsr-1h has no milestones/ dir. If you want 1h milestones, set "
        f"CKPT_DATASET = 'ckpt-xlsr-1h-traj'.")
assert os.path.exists(f"{MODEL_DIR}/config.json"), \
    f"no config.json under {MODEL_DIR} -- did NB-01 finish and publish this rung?"
print("checkpoint:", MODEL_DIR)

processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
for p in model.parameters(): p.requires_grad_(False)
model.config.apply_spec_augment = False      # no masking noise during attacks

DO_NORMALIZE = bool(getattr(processor.feature_extractor, "do_normalize", True))
BLANK_ID = model.config.pad_token_id
print(f"do_normalize {DO_NORMALIZE} | blank {BLANK_ID} | vocab {model.config.vocab_size}")

SPLITS = next((p for p in inputs if "splits" in p.lower()), None)
assert SPLITS, (f"Attach bn-asr-splits. Visible: {[os.path.basename(p) for p in inputs]}")
# Which utterances to attack. The NB-03 set is used for BOTH edit types when it
# is attached: T1 vs T5 is only a paired comparison if both run on the same audio.
# attack_eval_25.csv has 0 standalone-"না" utterances (verified 2026-09-25), so T1
# can never use it.
assert EVAL_SET in ("t5_frozen", "nb03"), f"EVAL_SET={EVAL_SET!r} is not valid"
_frozen = f"{SPLITS}/attack_eval_{N_EVAL}.csv"
_nb03 = next((f"{p}/attack_eval_t1.csv" for p in inputs
              if os.path.exists(f"{p}/attack_eval_t1.csv")), None)
if EVAL_SET == "nb03":
    assert _nb03, (
        "EVAL_SET='nb03' but attack_eval_t1.csv is not attached. Attach "
        "bn-attack-targets (run NB-03 first).\n"
        f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
    EV_CSV = _nb03
else:
    assert EDIT_TYPE != "T1", (
        "EVAL_SET='t5_frozen' cannot run T1: attack_eval_25.csv has zero "
        "standalone-'na' utterances (verified 2026-09-25). Use EVAL_SET='nb03'.")
    assert os.path.exists(_frozen), f"{_frozen} missing -- attach bn-asr-splits"
    EV_CSV = _frozen
    if _nb03:
        print("note: bn-attack-targets is attached but EVAL_SET='t5_frozen', so it is "
              "IGNORED -- both the utterance set and the targets come from "
              "attack_eval_25.csv. Deliberate: attaching a dataset must never "
              "change which utterances get attacked.")
print(f"eval set [{EVAL_SET}]: {EV_CSV}")
ev = pd.read_csv(EV_CSV)
PROV["attack_eval_csv"] = os.path.basename(EV_CSV)
PROV["fixed_eps"] = FIXED_EPS
PROV["ckpt_dataset"] = _DS
PROV["bisect_rounds"] = BISECT_ROUNDS

# ---------------------------------------------------------------------------
# Re-resolve the audio root, same hazard as NB-01: NB-00 baked in the mount
# point from ITS session. Here the whole attack set is only 25 files, so check
# every one of them rather than sampling.
# ---------------------------------------------------------------------------
def resolve_audio(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0):
        return df, os.path.dirname(p0)
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    cands = [d for pat in (f"/kaggle/input/*/{stem}",
                           f"/kaggle/input/*/*/{stem}",
                           f"/kaggle/input/*/*/*/{stem}")
             for d in glob.glob(pat) if os.path.isdir(d)]
    assert cands, (f"Audio directory '{stem}' not found under /kaggle/input.\n"
                   "Attach: + Add Input -> Competitions -> 'bengaliai speech'.\n"
                   f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
    df = df.copy()
    df["path"] = cands[0] + "/" + df["utt_id"].astype(str) + ext
    return df, cands[0]

ev, AUDIO_ROOT = resolve_audio(ev)
absent = [p for p in ev["path"] if not os.path.exists(p)]
assert not absent, f"{len(absent)}/{len(ev)} attack audio files missing, e.g.\n  {absent[0]}"
PROV["model_dir"] = MODEL_DIR; PROV["audio_root"] = AUDIO_ROOT
print(f"audio root: {AUDIO_ROOT}")
print(f"attack set: {len(ev)} utterances, all present")

## 4 · Build the targets

If NB-03's hand-built corpus is attached, use it. Otherwise generate automatically:

* **T1** — delete `না` from the reference transcript. Post-verbal, short, unstressed; one
  syllable inverts the meaning.
* **T5** — a *different real Bengali sentence* drawn from the corpus with a **matched token
  count**. This matters: NB-04's T5 target was 9 tokens against a ~55-character source, and
  forcing a much shorter output is plausibly easier, which would flatter T5 and bias against
  the T1 hypothesis. Matching length removes that.

In [ ]:
ZW = dict.fromkeys(map(ord, "‌‍"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()
def ntok(s): return len(processor.tokenizer(norm_bn(s)).input_ids)

# Use the resolved `inputs` list from the cell above, NOT a flat glob. Your own
# datasets mount at /kaggle/input/datasets/<user>/<name>, three levels deep, so
# glob("/kaggle/input/*") sees only the wrapper dirs. That is the same failure
# NB-01 hit, and on 2026-09-25 it silently hid an attached bn-attack-targets:
# T5 then auto-generated targets on the wrong utterance set and still "succeeded".
# The TARGET source is tied to EVAL_SET so the two can never diverge. Under
# "t5_frozen" the NB-03 corpus is not consulted even when attached: its utt_ids
# are disjoint from attack_eval_25.csv, so a merge would inner-join to zero rows.
TGT_CSV = None
if EVAL_SET == "nb03":
    TGT_CSV = next((f"{p}/bn_attack_targets.csv" for p in inputs
                    if os.path.exists(f"{p}/bn_attack_targets.csv")), None)
    assert TGT_CSV, (
        "EVAL_SET='nb03' but bn_attack_targets.csv not found. Attach "
        "bn-attack-targets (run NB-03 first).\n"
        f"Visible inputs: {[os.path.basename(p) for p in inputs]}")

if TGT_CSV:
    tg = pd.read_csv(TGT_CSV)
    tg = tg[tg.edit_type == EDIT_TYPE]
    ev = ev.merge(tg[["source_utterance_id", "target_text"]],
                  left_on="utt_id", right_on="source_utterance_id", how="inner")
    print(f"using NB-03 hand-built corpus: {len(ev)} pairs")
else:
    print("targets: AUTO-GENERATED from test.csv, RandomState(1337) -- CORRECT for")
    print("         NB-05b. The eight 25h/100h baseline trajectory conditions were")
    print("         measured this way, so matching them is the whole point. Do NOT")
    print("         substitute the NB-03 hand corpus here: its utterances are")
    print("         DISJOINT from attack_eval_25.csv and the preflight gate will")
    print("         refuse the run.\n")
    pool = pd.read_csv(f"{SPLITS}/test.csv")
    pool["ntok"] = pool["text"].map(ntok)
    ev["src_ntok"] = ev["text"].map(ntok)

    rows = []
    rng = np.random.RandomState(SEED)
    for _, r in ev.iterrows():
        srct = norm_bn(r["text"])
        if EDIT_TYPE == "T1":
            # STANDALONE particle only. A substring test is wrong: "না" occurs
            # inside many common Bengali words -- নাম (name), নারী (woman),
            # জানা (to know) -- and str.replace() would carve syllables out of
            # them, producing a target that is not a negation inversion at all.
            # Remove the LAST standalone "না", which is the post-verbal negation.
            toks = srct.split()
            if "না" not in toks: continue
            j = len(toks) - 1 - toks[::-1].index("না")
            tgt = norm_bn(" ".join(toks[:j] + toks[j+1:]))
            if tgt == srct or not tgt: continue
        else:
            cand = pool[(pool.utt_id != r["utt_id"]) &
                        (pool.ntok.between(r["src_ntok"] - 2, r["src_ntok"] + 2))]
            if len(cand) == 0: continue
            tgt = norm_bn(cand.sample(1, random_state=int(rng.randint(1e6))).iloc[0]["text"])
        rows.append({**r.to_dict(), "target_text": tgt})
    ev = pd.DataFrame(rows)
    if EDIT_TYPE == "T1":
        # NB-00 froze the attack set using a SUBSTRING test for "না", so the
        # advertised "15 with না" overcounts utterances that actually carry the
        # standalone negation particle. Find out before spending GPU.
        print(f"\nT1: {len(ev)} of {N_EVAL} utterances carry a standalone না")
        assert len(ev) >= 10, (
            f"only {len(ev)} T1-attackable utterances -- too few for RQ2. "
            f"Either build the NB-03 corpus, or freeze an additional T1-specific "
            f"attack subset from the test split (without touching bn-asr-splits).")

ev["target_text"] = ev["target_text"].map(norm_bn)
ev["target_tokens"] = ev["target_text"].map(ntok)
ev["source_tokens"] = ev["text"].map(ntok)
ev = ev.reset_index(drop=True)
PROV["n_attackable"] = int(len(ev))
PROV["n_eval_frozen"] = int(N_EVAL)
PROV["mean_target_tokens"] = float(ev.target_tokens.mean())
PROV["mean_source_tokens"] = float(ev.source_tokens.mean())
print(f"{EDIT_TYPE}: {len(ev)} attackable utterances")
print(f"target tokens  mean {ev.target_tokens.mean():.1f}  vs source {ev.source_tokens.mean():.1f}")
print("  (these should be close for T5; T1 is by construction slightly shorter)")
ev[["utt_id", "source_tokens", "target_tokens", "text", "target_text"]].head()

## 4b · Preflight gate — would this run be comparable to the baseline eight?

In [ ]:
# ============================================================================
#  PREFLIGHT GATE
# ============================================================================
_fail = []
if os.path.basename(EV_CSV) != "attack_eval_25.csv":
    _fail.append(f"eval set is {os.path.basename(EV_CSV)!r}; survival must sit on the "
                 f"same 25 utterances as the RQ1 ladder. Detach bn-attack-targets.")
if len(ev) != 25:
    _fail.append(f"{len(ev)} utterances loaded, expected 25.")
if TGT_CSV is not None:
    _fail.append(f"targets came from {os.path.basename(TGT_CSV)}; need the auto-generated "
                 f"targets the ladder used.")
if f"ckpt-xlsr-{RUNG}h" not in MODEL_DIR:
    _fail.append(f"MODEL_DIR={MODEL_DIR!r} does not match rung {RUNG}.")
if "traj" in MODEL_DIR:
    _fail.append(f"MODEL_DIR={MODEL_DIR!r} is a trajectory dataset. Use the published rung.")
if "milestone" in MODEL_DIR:
    _fail.append(f"MODEL_DIR={MODEL_DIR!r} is a milestone. RQ4 uses rung finals.")
if BISECT_ROUNDS != 0:
    _fail.append(f"BISECT_ROUNDS={BISECT_ROUNDS}, expected 0 under FIXED_EPS.")
assert not _fail, "PREFLIGHT FAILED:\n  - " + "\n  - ".join(_fail)

# ffmpeg is required for the codec transforms. Fail here, not 20 minutes in.
import shutil, subprocess
FFMPEG = shutil.which("ffmpeg")
assert FFMPEG, ("ffmpeg not found. The codec transforms cannot run. Install it in the "
                "pip cell (`apt-get install -y ffmpeg`) or drop the 'codec' entries "
                "from TRANSFORMS -- but say so in the paper if you do.")
_enc = subprocess.run([FFMPEG, "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
_need = {"mp3": "libmp3lame", "aac": "aac", "opus": "libopus"}
SKIP = set()
for nm, kind, par in TRANSFORMS:
    if kind == "codec" and _need[par[0]] not in _enc:
        SKIP.add(nm)
print(f"ffmpeg OK: {FFMPEG}")
for nm, kind, par in TRANSFORMS:
    print(f"  {nm:12s} {'SKIPPED (encoder missing)' if nm in SKIP else 'ready'}")
assert len(SKIP) < len([t for t in TRANSFORMS if t[1] == "codec"]), \
    "every codec encoder is missing; ffmpeg build is unusable for this experiment"

print("\nPREFLIGHT PASSED")
print(f"  rung {RUNG}h  |  {os.path.basename(EV_CSV)} ({len(ev)} utts)  |  "
      f"budget {EPS_HI:.1e}")
print(f"  exposure for this rung: "
      f"{ {1:0.98, 5:4.90, 25:24.60, 100:98.02}[RUNG] } h distinct audio")
print("  survival will be reported against that exposure in NB-09.")


## 5 · Batched differentiable front-end and loss

In [ ]:
SR = 16000

def load_batch(rows):
    """Length-bucketed batch. Returns padded waveforms + a validity mask."""
    wavs = []
    for _, r in rows.iterrows():
        a, _ = librosa.load(r["path"], sr=SR, mono=True)
        a = np.asarray(a, np.float32)
        pk = np.abs(a).max()
        wavs.append(a / max(pk, 1.0))
    T = max(len(w) for w in wavs)
    x = torch.zeros(len(wavs), T, dtype=torch.float32)
    m = torch.zeros(len(wavs), T, dtype=torch.float32)
    for i, w in enumerate(wavs):
        x[i, :len(w)] = torch.from_numpy(w); m[i, :len(w)] = 1.0
    return x.to(DEVICE), m.to(DEVICE)


def frontend_b(x, mask):
    """Masked zero-mean/unit-variance. Computing stats over the PADDING would
    corrupt short utterances in a mixed-length batch."""
    if DO_NORMALIZE:
        n = mask.sum(-1, keepdim=True).clamp(min=1)
        mu = (x * mask).sum(-1, keepdim=True) / n
        var = (((x - mu) * mask) ** 2).sum(-1, keepdim=True) / n
        x = (x - mu) / torch.sqrt(var + 1e-7)
        x = x * mask
    return x


def logits_b(x, mask):
    return model(input_values=frontend_b(x, mask),
                 attention_mask=mask.long()).logits


def ctc_per_sample(x, mask, tgt_pad, tgt_len):
    """Per-sample CTC loss -> lets us mask out finished samples."""
    lg = logits_b(x, mask)
    logp = F.log_softmax(lg.float(), -1).transpose(0, 1)          # (L,B,V)
    in_len = model._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
    in_len = in_len.clamp(max=logp.shape[0])
    return F.ctc_loss(logp, tgt_pad, in_len, tgt_len,
                      blank=BLANK_ID, zero_infinity=True, reduction="none")


def decode_b(x, mask):
    with torch.no_grad():
        ids = torch.argmax(logits_b(x, mask), -1)
    return [norm_bn(s) for s in processor.batch_decode(ids)]


def int16_rt(x):
    return torch.clamp((x * 32767.0).round(), -32768.0, 32767.0) / 32767.0

def snr_db_b(clean, adv, mask):
    n = (adv - clean) * mask
    s = clean * mask
    return (10 * torch.log10(s.pow(2).sum(-1) / (n.pow(2).sum(-1) + 1e-12))).cpu().numpy()

## 6 · Batched PGD with per-sample budgets

Every utterance in the batch carries its **own** eps, its own success flag and its own
best-so-far delta. Finished samples are masked out of the loss so the optimiser stops spending
capacity on them, and their delta is frozen at the moment of success.

In [ ]:
def pgd_batch(x, mask, tgt_pad, tgt_len, targets, eps_vec,
              steps, delta_init=None, lr=LR):
    """eps_vec: (B,) per-sample L-inf budgets.
       Returns (success bool array, frozen deltas, steps_used)."""
    B = x.shape[0]
    eps_col = eps_vec.view(-1, 1)
    delta = (torch.zeros_like(x) if delta_init is None
             else torch.max(torch.min(delta_init.clone(), eps_col), -eps_col))
    delta.requires_grad_(True)
    opt = torch.optim.Adam([delta], lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    active = torch.ones(B, dtype=torch.bool, device=DEVICE)
    frozen = torch.zeros_like(x)
    success = np.zeros(B, dtype=bool)
    best_loss = torch.full((B,), float("inf"), device=DEVICE)
    stall = torch.zeros(B, dtype=torch.long, device=DEVICE)

    for step in range(steps):
        if not active.any():
            break
        x_adv = torch.clamp(x + delta, -1, 1) * mask
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            per = ctc_per_sample(x_adv, mask, tgt_pad, tgt_len)
        loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)

        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt)
        if delta.grad is not None:
            delta.grad[~active] = 0          # finished samples stop moving
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            delta.data = torch.max(torch.min(delta.data, eps_col), -eps_col)

        with torch.no_grad():                # stall detection
            improved = per < best_loss - 1e-3
            best_loss = torch.minimum(best_loss, per.detach())
            stall = torch.where(improved, torch.zeros_like(stall), stall + 1)
            give_up = active & (stall > STALL_PATIENCE)
            active = active & ~give_up

        if (step + 1) % CHECK_EVERY == 0:
            hyp = decode_b(int16_rt(torch.clamp(x + delta, -1, 1)) * mask, mask)
            with torch.no_grad():
                for i in range(B):
                    if active[i] and hyp[i] == targets[i]:
                        frozen[i] = delta[i].detach()
                        success[i] = True
                        active[i] = False
    return success, frozen, step + 1

## 7 · Per-sample geometric bisection, run in lockstep

In [ ]:
def eps50_batch(x, mask, tgt_pad, tgt_len, targets, verbose=True):
    """Per-sample geometric bisection for the smallest budget that still works.

    ---------------------------------------------------------------------------
    BUG FIXED 2026-09-22 (found in the rung-1 T5 run, 40% censoring).

    The previous version ended with

        ok, best, _ = pgd_batch(..., hi, CONFIRM_STEPS, delta_init=best)

    which REASSIGNED `best` wholesale to pgd_batch's return value. That return
    is a `frozen` buffer holding zeros for any sample that did not succeed on
    that particular pass -- so one unlucky final pass wiped out a delta that had
    already been verified at a larger budget. Those samples were then written
    out as censored, with SNR 70-86 dB (the signature of delta == 0) even though
    the ceiling probe had already proved them attackable.

    In the rung-1 run that discarded ~6 of 25 real measurements. Worse, it does
    so at a RATE THAT VARIES BY MODEL, so comparing median eps50 across rungs
    would have been comparing differently-biased subsamples -- straight into
    RQ1's headline.

    Fix: keep a running `best_eps` / `best_delta` that is only ever updated on a
    VERIFIED success. Nothing overwrites a confirmed result. Censoring now means
    exactly one thing: the attack failed at every budget tried, ceiling included.

    Side benefit: eps50 becomes a guaranteed UPPER BOUND on the true minimum.
    Optimiser noise can only make it conservative, never wrong in an
    unpredictable direction -- which is the property you want in a metric you
    are about to compare across four models.
    ---------------------------------------------------------------------------
    """
    B = x.shape[0]
    lo = torch.full((B,), EPS_LO, device=DEVICE)
    hi = torch.full((B,), EPS_HI, device=DEVICE)
    best_eps   = torch.full((B,), float("nan"), device=DEVICE)   # verified only
    best_delta = torch.zeros_like(x)

    def record(okt, eps_vec, d):
        """Adopt a result only when it is both successful and better."""
        nonlocal best_eps, best_delta
        better = okt & (torch.isnan(best_eps) | (eps_vec < best_eps))
        best_eps   = torch.where(better, eps_vec, best_eps)
        best_delta = torch.where(better.view(-1, 1), d, best_delta)

    # Ceiling probe. Failure here is genuine right-censoring.
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, hi, CONFIRM_STEPS)
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, hi, d)
    feasible = ok.copy()
    if verbose:
        print(f"  ceiling probe eps={EPS_HI:.1e}: {feasible.sum()}/{B} feasible, "
              f"{(~feasible).sum()} censored")

    for rnd in range(BISECT_ROUNDS):
        mid = torch.sqrt(lo * hi)
        ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                             PROBE_STEPS, delta_init=best_delta)
        ok = ok & feasible
        okt = torch.tensor(ok, device=DEVICE)
        hi = torch.where(okt, mid, hi)
        lo = torch.where(okt, lo, mid)
        record(okt, mid, d)
        if verbose:
            print(f"  round {rnd+1}/{BISECT_ROUNDS}: {ok.sum()}/{B} succeeded "
                  f"| median eps {mid.median().item():.3e}")

    # Budget-consistency round: one more bisection at the FULL step budget, so
    # the reported boundary is measured at the budget eps50 is defined at rather
    # than at the cheaper probe budget. See RUNLOG for why this matters.
    #
    # Skipped in FIXED_EPS mode: there is no bracket to refine. The ceiling probe
    # already ran at CONFIRM_STEPS, so the single reported budget is measured at
    # the full budget and needs no second pass.
    if BISECT_ROUNDS == 0:
        eps       = best_eps.cpu().numpy()
        confirmed = ~np.isnan(eps)
        if verbose:
            print(f"  -> FIXED_EPS {EPS_HI:.1e}: {confirmed.sum()}/{B} succeeded, "
                  f"{(~confirmed).sum()} resisted")
        return eps, np.ones(B), confirmed, best_delta

    mid = torch.sqrt(lo * hi)
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                         CONFIRM_STEPS, delta_init=best_delta)
    ok = ok & feasible
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, mid, d)
    if verbose:
        print(f"  budget-consistency round @ {CONFIRM_STEPS} steps: {ok.sum()}/{B}")

    eps       = best_eps.cpu().numpy()
    confirmed = ~np.isnan(eps)                 # censored == never succeeded
    width     = (hi / lo).cpu().numpy()
    if verbose:
        print(f"  -> {confirmed.sum()}/{B} measured, {(~confirmed).sum()} censored")
    return eps, width, confirmed, best_delta


## 8 · Validate the batching before trusting it

Batching is a substantial rewrite. If masked normalisation or the per-sample CTC lengths are
wrong, results would be quietly biased rather than obviously broken. So: run two utterances
batched, then the same two alone, and compare.

> **This costs about three extra eps50 searches (~1–2 GPU-h).** Run it once, for
> `(RUNG=1, EDIT_TYPE="T5")`, record the agreement in `RUNLOG.md`, then set
> `VALIDATE_BATCHING = False` for the other seven runs. It is a one-off correctness check on
> the batching code, and that code does not change between runs.

In [ ]:
if VALIDATE_BATCHING and len(ev) >= 2:
    sub = ev.head(2)
    xb, mb = load_batch(sub)
    tp = [torch.tensor(processor.tokenizer(t).input_ids, device=DEVICE) for t in sub.target_text]
    tl = torch.tensor([len(t) for t in tp], device=DEVICE)
    tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
    tgts = list(sub.target_text)

    print("batched (2 together):")
    t0 = time.time(); e_b, _, c_b, _ = eps50_batch(xb, mb, tpad, tl, tgts); tb = time.time()-t0

    print("\nsingle (one at a time):")
    e_s = []
    t0 = time.time()
    for i in range(2):
        xs, ms = xb[i:i+1], mb[i:i+1]
        e, _, c, _ = eps50_batch(xs, ms, tpad[i:i+1], tl[i:i+1], [tgts[i]])
        e_s.append(e[0])
    ts = time.time()-t0

    print(f"\n{'':6s} {'batched':>12s} {'single':>12s}  {'rel diff':>9s}")
    for i in range(2):
        a, b = e_b[i], e_s[i]
        rd = "n/a" if (np.isnan(a) or np.isnan(b)) else f"{100*abs(a-b)/b:8.1f}%"
        print(f"  #{i}  {a:12.3e} {b:12.3e}  {rd:>9s}")
    print(f"\nwall-clock: batched {tb:.0f}s vs single {ts:.0f}s -> {ts/tb:.2f}x")
    print("Agreement within ~15% is fine (PGD is stochastic). Larger means a batching bug.")

## 8b · Is eps50 converged at this step budget?


In [ ]:
# ---------------------------------------------------------------------------
# CONVERGENCE CHECK.
#
# eps50 is defined relative to CONFIRM_STEPS. If doubling the budget moves it
# materially, the metric is still tracking optimiser convergence rather than
# model robustness -- and since rungs may converge at different rates, that
# would leak straight into the RQ1 comparison. Three utterances is enough to
# detect it.
# ---------------------------------------------------------------------------
if len(ev) >= 3:
    sub = ev.head(3)
    xb, mb = load_batch(sub)
    tp = [torch.tensor(processor.tokenizer(t).input_ids, device=DEVICE) for t in sub.target_text]
    tl = torch.tensor([len(t) for t in tp], device=DEVICE)
    tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
    tgts = list(sub.target_text)

    _saved = CONFIRM_STEPS
    print(f"at CONFIRM_STEPS = {_saved}:")
    e1, _, _, _ = eps50_batch(xb, mb, tpad, tl, tgts, verbose=False)
    CONFIRM_STEPS = _saved * 2
    print(f"at CONFIRM_STEPS = {CONFIRM_STEPS}:")
    e2, _, _, _ = eps50_batch(xb, mb, tpad, tl, tgts, verbose=False)
    CONFIRM_STEPS = _saved

    print(f"\n{'utt':>4s} {'@'+str(_saved):>12s} {'@'+str(_saved*2):>12s} {'shift':>9s}")
    shifts = []
    for i in range(len(sub)):
        a, b = e1[i], e2[i]
        if np.isnan(a) or np.isnan(b):
            print(f"{i:>4d} {'censored':>12s}"); continue
        sh = abs(a - b) / a; shifts.append(sh)
        print(f"{i:>4d} {a:12.3e} {b:12.3e} {100*sh:8.1f}%")
    if shifts:
        ms = float(np.median(shifts))
        print(f"\nmedian shift on doubling the budget: {100*ms:.1f}%")
        if ms < 0.10:
            print("Converged. eps50 is stable at this budget -- safe to compare rungs.")
        else:
            print("NOT CONVERGED. Raise CONFIRM_STEPS and re-run, or state explicitly")
            print("that eps50 is budget-relative and hold the budget fixed across ALL")
            print("rungs so the comparison stays internally valid.")
        PROV["convergence_shift_pct"] = round(100 * ms, 1)


## 4c · The transform chain

In [ ]:
# ============================================================================
#  Deployment transforms. Each takes a float waveform in [-1, 1] at 16 kHz and
#  returns one of the same length, so the mask and the decoder are unaffected.
#
#  Every transform round-trips through int16 first, because that is what a real
#  pipeline receives. Skipping that step would flatter the attack: a float32
#  waveform carries perturbation detail no deployed chain ever sees.
# ============================================================================
# math is NOT imported by the boilerplate cell (only os/sys/json/time/random/re/
# glob/unicodedata/numpy/pandas/torch/subprocess/transformers/librosa/shutil), and
# _align and t_noise both need it. Importing here keeps this cell self-contained.
import math, tempfile, subprocess, wave

SR = 16000

def _to_int16(x):
    return np.clip(np.asarray(x, dtype=np.float64) * 32767.0, -32768, 32767).astype(np.int16)

def _to_float(x):
    return np.asarray(x, dtype=np.float32) / 32767.0

def _write_wav(path, x, sr=SR):
    with wave.open(path, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes(_to_int16(x).tobytes())

def _read_wav(path):
    with wave.open(path, "rb") as w:
        n = w.getnframes(); sr = w.getframerate()
        raw = w.readframes(n)
    return _to_float(np.frombuffer(raw, dtype=np.int16)), sr

def _align(y, x, max_lag=8000):
    """Remove codec encoder delay before comparing y to x.

    MEASURED 2026-09-27, and this is not optional. ffmpeg's AAC encoder delays
    the signal by 3424 samples at 16 kHz -- 214 ms. Comparing the decoded
    waveform at lag 0 gave a relative error of 0.775; aligning first gave
    0.0008. Without this step every AAC row would report the attack destroyed,
    caused entirely by a time shift rather than by any codec damage. MP3 delays
    by ~1100-2400 samples and Opus by ~312, so all of them need it.

    Alignment removes only the shift. Amplitude and spectral damage survive it,
    which is what we want to measure.
    """
    x = np.asarray(x, dtype=np.float64); y = np.asarray(y, dtype=np.float64)
    n = len(x)
    if len(y) < n + max_lag:
        y = np.pad(y, (0, n + max_lag - len(y)))
    L = 1 << int(math.ceil(math.log2(len(y) + n)))
    cc = np.fft.irfft(np.fft.rfft(y, L) * np.conj(np.fft.rfft(x, L)), L)
    lag = int(np.argmax(cc[:max_lag + 1]))
    return y[lag:lag + n].astype(np.float32)

def t_codec(x, fmt, bitrate):
    """Encode to a lossy codec and decode back, via ffmpeg."""
    with tempfile.TemporaryDirectory() as d:
        src = f"{d}/in.wav"; enc = f"{d}/enc.{'ogg' if fmt=='opus' else fmt}"
        out = f"{d}/out.wav"
        _write_wav(src, x)
        cod = {"mp3": "libmp3lame", "aac": "aac", "opus": "libopus"}[fmt]
        for cmd in ([FFMPEG,"-y","-loglevel","error","-i",src,"-c:a",cod,"-b:a",bitrate,enc],
                    [FFMPEG,"-y","-loglevel","error","-i",enc,"-ar",str(SR),"-ac","1",out]):
            r = subprocess.run(cmd, capture_output=True, text=True)
            if r.returncode != 0:
                raise RuntimeError(f"ffmpeg failed: {r.stderr.strip()[:200]}")
        y, sr = _read_wav(out)
    assert sr == SR, f"codec returned {sr} Hz"
    return _align(y, x)

def t_resample(x, target_sr):
    """Round-trip through a lower sample rate: 16k -> target -> 16k, via ffmpeg
    so the anti-alias filtering is a real implementation, not a toy."""
    with tempfile.TemporaryDirectory() as d:
        src = f"{d}/in.wav"; low = f"{d}/low.wav"; out = f"{d}/out.wav"
        _write_wav(src, x)
        for cmd in ([FFMPEG,"-y","-loglevel","error","-i",src,"-ar",str(target_sr),low],
                    [FFMPEG,"-y","-loglevel","error","-i",low,"-ar",str(SR),out]):
            r = subprocess.run(cmd, capture_output=True, text=True)
            if r.returncode != 0:
                raise RuntimeError(f"ffmpeg resample failed: {r.stderr.strip()[:200]}")
        y, _ = _read_wav(out)
    return _align(y, x)

def t_mulaw(x, _=None):
    """G.711 mu-law companding, 8-bit. Standard telephony quantisation."""
    mu = 255.0
    s = np.sign(x)
    y = s * np.log1p(mu * np.abs(x)) / np.log1p(mu)          # compress
    q = np.round((y + 1.0) * 127.5) / 127.5 - 1.0             # 8-bit quantise
    s2 = np.sign(q)
    return (s2 * ((1.0 + mu) ** np.abs(q) - 1.0) / mu).astype(np.float32)   # expand

def t_noise(x, snr_db):
    """Additive white Gaussian noise at a fixed SNR relative to THIS waveform.
    Seeded per call so the run reproduces."""
    rng = np.random.RandomState(1337)
    p_sig = float(np.mean(np.asarray(x, dtype=np.float64) ** 2))
    if p_sig <= 0: return np.asarray(x, dtype=np.float32)
    p_noise = p_sig / (10.0 ** (snr_db / 10.0))
    return (np.asarray(x) + rng.randn(len(x)) * math.sqrt(p_noise)).astype(np.float32)

def apply_transform(x, kind, par):
    if kind == "codec":    return t_codec(x, par[0], par[1])
    if kind == "resample": return t_resample(x, par)
    if kind == "mulaw":    return t_mulaw(x)
    if kind == "noise":    return t_noise(x, par)
    raise ValueError(kind)

# --- self-test on one real clip before spending GPU time -------------------
# A transform that silently returns its input would report 100% survival and
# look like a finding. Check each one actually changes the waveform, and that
# the clean transcript is still mostly recoverable afterwards -- a transform
# that destroys the speech tells us nothing about the attack.
_x, _m = load_batch(ev.iloc[:1])
_x0 = (_x[0] * _m[0]).detach().cpu().numpy().astype(np.float64)
_ref = norm_bn(ev.iloc[0]["text"])
print("transform self-test on one clip:")
print(f"{'transform':12s} {'rel.change':>11s} {'clean CER after':>16s}")
_bad = []
for nm, kind, par in TRANSFORMS:
    if nm in SKIP: continue
    try:
        _y = apply_transform(_x0, kind, par)
    except Exception as e:
        _bad.append(f"{nm}: {type(e).__name__}: {e}"); continue
    _rel = float(np.linalg.norm(_y - _x0) / (np.linalg.norm(_x0) + 1e-12))
    _t = torch.tensor(_y, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    _h = decode_b(_t, _m[:1])[0]
    try:
        import jiwer; _cer = jiwer.cer(_ref, _h) if _ref.strip() else float("nan")
    except Exception: _cer = float("nan")
    print(f"{nm:12s} {_rel:11.4f} {_cer:16.3f}")
    if _rel < 1e-6: _bad.append(f"{nm}: waveform unchanged -- transform is a no-op")
    # A relative change above ~0.5 on CLEAN speech means the waveform is
    # misaligned, not damaged: real codec damage at these bitrates is a few
    # percent. Catch it here rather than reporting the attack destroyed.
    if _rel > 0.5:
        _bad.append(f"{nm}: rel.change {_rel:.3f} on clean speech -- almost certainly a "
                    f"residual time shift that _align did not remove. Raise max_lag.")
assert not _bad, "transform self-test failed:\n  - " + "\n  - ".join(_bad)
print("\nall transforms change the waveform, stay aligned, and still decode.")
del _x, _m, _x0
torch.cuda.empty_cache()


## 9 · The main sweep

In [ ]:
ev = ev.sort_values("duration").reset_index(drop=True)   # length buckets = less padding
OUT = f"{WORK}/eps50_rung{RUNG}_{CHECKPOINT}_{EDIT_TYPE}.csv"

# Resume: if a previous session ran out of time, attach its output dataset as an
# input and the utterances it already measured are carried over, not redone.
#
# Two fixes over NB-05 (2026-09-26):
#  (a) the old glob was "/kaggle/input/*/<file>", only TWO levels deep, while your
#      own datasets mount at /kaggle/input/datasets/<user>/<name> -- three levels.
#      So resume silently never fired. Same defect class as the flat-glob bug
#      already fixed in section 3; it survived here. Use the resolved `inputs`.
#  (b) the superseded 38-utt runs wrote a CSV with the SAME basename
#      (eps50_rung1_<ckpt>_T5.csv). Attaching one of those would have pulled 38
#      foreign utterances into `results` and contaminated the output, because the
#      preflight gate has already run by this point. Rows are now checked against
#      the current attack set and rejected if they do not belong to it.
results = []
for _p in inputs:
    _f = f"{_p}/{os.path.basename(OUT)}"
    if not os.path.exists(_f):
        continue
    prev = pd.read_csv(_f)
    _foreign = set(prev.utt_id) - set(ev.utt_id)
    assert not _foreign, (
        f"{os.path.basename(_p)} holds {len(prev)} rows, {len(_foreign)} of which are "
        f"NOT in the current 25-utterance attack set. That dataset was almost "
        f"certainly produced on the 38-utt NB-03 set (the superseded "
        f"eps50-rung1-* runs). Detach it -- resuming from it would mix two "
        f"disjoint utterance sets into one CSV.")
    results.extend(prev.to_dict("records"))
    print(f"resuming: {len(prev)} utterances already measured in {os.path.basename(_p)}")
done = {r["utt_id"] for r in results}
todo = ev[~ev.utt_id.isin(done)].reset_index(drop=True)

batches = [todo.iloc[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
print(f"{len(todo)} utterances remaining -> {len(batches)} batches of up to {BATCH_SIZE}\n")

DEADLINE = time.time() + SESSION_HOURS * 3600
t_start = time.time()

for bi, sub in enumerate(batches):
    if time.time() > DEADLINE:
        print(f"*** SESSION BUDGET ({SESSION_HOURS} h) reached. "
              f"{sum(len(b) for b in batches[bi:])} utterances not yet measured. ***\n")
        break

    print(f"=== batch {bi+1}/{len(batches)} ({len(sub)} utts, "
          f"{sub.duration.min():.1f}-{sub.duration.max():.1f}s) ===")
    xb, mb = load_batch(sub)
    tp = [torch.tensor(processor.tokenizer(t).input_ids, device=DEVICE) for t in sub.target_text]
    tl = torch.tensor([len(t) for t in tp], device=DEVICE)
    tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
    tgts = list(sub.target_text)

    t0 = time.time()
    eps, width, conf, best = eps50_batch(xb, mb, tpad, tl, tgts)
    dt = time.time() - t0

    x_adv = int16_rt(torch.clamp(xb + best, -1, 1)) * mb
    snrs = snr_db_b(xb, x_adv, mb)
    hyp = decode_b(x_adv, mb)

    # ---- RQ4: push each adversarial waveform through the deployment chain ----
    # Only utterances whose attack VERIFIED on the raw waveform can be tested:
    # survival is conditional on having something to survive. Censored rows get
    # None for every transform so the distinction stays visible downstream.
    _ok_i = [i for i in range(len(sub)) if not np.isnan(eps[i])]
    surv = {nm: [None] * len(sub) for nm, _, _ in TRANSFORMS}
    if _ok_i:
        _xa = (x_adv * mb).detach().cpu().numpy().astype(np.float64)
        _len = mb.sum(dim=1).detach().cpu().numpy().astype(int)
        for nm, kind, par in TRANSFORMS:
            if nm in SKIP:
                continue
            _batch = torch.zeros_like(x_adv)
            for i in _ok_i:
                n = int(_len[i])
                y = apply_transform(_xa[i][:n], kind, par)
                _batch[i, :n] = torch.tensor(y, dtype=_batch.dtype, device=DEVICE)
            _h = decode_b(int16_rt(torch.clamp(_batch, -1, 1)) * mb, mb)
            for i in _ok_i:
                tgt = norm_bn(sub.iloc[i]["target_text"])
                ref = norm_bn(sub.iloc[i]["text"])
                got = norm_bn(_h[i])
                surv[nm][i] = ("survives" if got == tgt
                               else "reverted" if got == ref
                               else "broken")
        _n = len(_ok_i)
        print("  survival on this batch (of "
              f"{_n} verified attack{'s' if _n != 1 else ''}):")
        for nm, _, _ in TRANSFORMS:
            if nm in SKIP: continue
            c = [surv[nm][i] for i in _ok_i]
            print(f"    {nm:12s} survives {c.count('survives'):2d}  "
                  f"broken {c.count('broken'):2d}  reverted {c.count('reverted'):2d}")

    # Write the waveforms once so no future experiment repeats this GPU cost.
    if SAVE_WAVS:
        _wd = f"{WORK}/wav"; os.makedirs(_wd, exist_ok=True)
        _xr = (xb * mb).detach().cpu().numpy().astype(np.float64)
        _xaS = (x_adv * mb).detach().cpu().numpy().astype(np.float64)
        _lenS = mb.sum(dim=1).detach().cpu().numpy().astype(int)
        for i, (_, r) in enumerate(sub.iterrows()):
            n = int(_lenS[i]); u = str(r["utt_id"])
            _write_wav(f"{_wd}/{u}_clean.wav", _xr[i][:n])
            if not np.isnan(eps[i]):
                _write_wav(f"{_wd}/{u}_adv.wav", _xaS[i][:n])

    for i, (_, r) in enumerate(sub.iterrows()):
        results.append(dict(
            utt_id=r["utt_id"], rung=RUNG, edit_type=EDIT_TYPE, checkpoint=CHECKPOINT,
            duration=r["duration"], source_text=r["text"], target_text=r["target_text"],
            source_tokens=int(r["source_tokens"]), target_tokens=int(r["target_tokens"]),
            eps50=float(eps[i]) if not np.isnan(eps[i]) else None,
            # SNR/decode are meaningful only where an attack was verified.
            # Writing them for censored rows produced the 70-86 dB values in
            # the rung-1 run, which looked like data but were delta == 0.
            censored=bool(np.isnan(eps[i])),
            eps50_per_token=(float(eps[i]) / r["target_tokens"]) if not np.isnan(eps[i]) else None,
            bracket_width=float(width[i]), snr_db=float(snrs[i]),
            decoded=(hyp[i] if not np.isnan(eps[i]) else None),
            exact_match=(bool(hyp[i] == r["target_text"])
                         if not np.isnan(eps[i]) else False),
            use_amp=USE_AMP, model_dir=MODEL_DIR,
            **{f"surv_{nm}": surv[nm][i] for nm, _, _ in TRANSFORMS},
        ))

    # Write after EVERY batch. The original only wrote at the very end, so a
    # session killed on the last batch threw away hours of finished work.
    pd.DataFrame(results).to_csv(OUT, index=False)

    remaining = len(batches) - (bi + 1)
    print(f"  batch done in {dt/60:.1f} min | {(~np.isnan(eps)).sum()}/{len(sub)} uncensored "
          f"| {len(results)} rows written")
    if remaining:
        print(f"  projected for the {remaining} remaining batches: "
              f"{remaining*dt/3600:.1f} h "
              f"({'fits' if time.time()+remaining*dt < DEADLINE else 'WILL NOT FIT this session'})\n")
    else:
        print()

print(f"TOTAL {(time.time()-t_start)/60:.1f} min this session | {len(results)}/{len(ev)} measured")

## 10 · Results and save

In [ ]:
CK_TAG = "final" if CHECKPOINT == "final" else "ms" + CHECKPOINT.split("-")[1]
if FIXED_EPS is not None:
    CK_TAG += "-fixedeps"        # keep trajectory runs separate from eps50 runs
# Every checkpoint of a rung gets its OWN dataset. Before this, 100h final and
# all three 100h milestones were told to publish as "eps50-rung100-T5" (the old
# NB-05 naming), so "New Version" on that dataset silently replaced one
# checkpoint's results with another's.
#
# NB-05b uses an "eps50b-" prefix so these 25-utterance runs land in NEW datasets
# and cannot version over the four superseded 38-utterance "eps50-rung1-*" ones,
# which are kept for the record. Choose "New Dataset" when publishing, not
# "New Version".
df = pd.DataFrame(results)
ok = df[~df.censored] if len(df) else df
complete = len(df) >= len(ev)

# Clean WER on the SAME attack utterances, measured here so a trajectory run
# needs no NB-02 pass. The figure plots attack success against clean WER and both
# numbers must come from the same checkpoint.
if FIXED_EPS is not None:
    try:
        import jiwer
        _wav, _msk = load_batch(ev)
        _hyp = decode_b(_wav, _msk)
        _ref = [norm_bn(t) for t in ev.text]
        _pr = [(r, h) for r, h in zip(_ref, _hyp) if r.strip()]
        PROV["clean_wer_attackset"] = float(jiwer.wer([p[0] for p in _pr], [p[1] for p in _pr]))
        PROV["clean_cer_attackset"] = float(jiwer.cer([p[0] for p in _pr], [p[1] for p in _pr]))
        print(f"clean WER on the attack set: {PROV['clean_wer_attackset']:.4f}  "
              f"CER {PROV['clean_cer_attackset']:.4f}   <- x-axis for the trajectory figure")
        del _wav, _msk; torch.cuda.empty_cache()
    except Exception as e:
        print(f"clean-WER measurement failed ({type(e).__name__}: {e}); "
              f"fall back to NB-02's clean_wer_all_checkpoints.csv")

print(f"=== RUNG {RUNG}h | {EDIT_TYPE}"
      + (f" | FIXED_EPS {FIXED_EPS:.1e} ===" if FIXED_EPS is not None else " ==="))
if FIXED_EPS is not None:
    print("  FIXED_EPS mode: every success is recorded at exactly the fixed budget, so")
    print("  eps50, its IQR and its bootstrap CI below are DEGENERATE and carry no")
    print("  information. Use ASR@eps and the per-utterance `censored` column only.")
print(f"utterances      : {len(df)} of {len(ev)}")
print(f"censored        : {df.censored.sum()} ({100*df.censored.mean():.0f}%)  <- report this")
if len(ok):
    print(f"median eps50    : {ok.eps50.median():.3e}  ({ok.eps50.median()*32767:.1f} int16 units)")
    print(f"IQR             : {ok.eps50.quantile(.25):.3e} - {ok.eps50.quantile(.75):.3e}")
    print(f"per target token: {ok.eps50_per_token.median():.3e}   <- length-normalised")
    print(f"median SNR      : {ok.snr_db.median():.1f} dB")
    print(f"exact match     : {ok.exact_match.sum()}/{len(ok)}")

    # A bracket width near 1.0 with eps50 at the floor means the search hit
    # EPS_LO: LEFT-censored, and just as biasing as the right-censored kind.
    at_floor = int(((ok.eps50 <= EPS_LO * 1.05)).sum())
    if at_floor:
        print(f"WARNING: {at_floor} utterances sit at the EPS_LO floor ({EPS_LO:.0e}) "
              "-- left-censored. Lower EPS_LO and re-run this rung.")

    b = [ok.eps50.sample(len(ok), replace=True).median() for _ in range(2000)]
    print(f"bootstrap 95% CI: [{np.percentile(b,2.5):.3e}, {np.percentile(b,97.5):.3e}]")

df.to_csv(f"{WORK}/eps50_rung{RUNG}_{CHECKPOINT}_{EDIT_TYPE}.csv", index=False)
PROV.update(n_utts=len(df), n_expected=len(ev), complete=bool(complete),
            n_censored=int(df.censored.sum()) if len(df) else 0,
            median_eps50=float(ok.eps50.median()) if len(ok) else None,
            median_eps50_per_token=float(ok.eps50_per_token.median()) if len(ok) else None,
            median_snr=float(ok.snr_db.median()) if len(ok) else None,
            minutes=round((time.time()-t_start)/60, 1))
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), indent=2)

KAGGLE_USER = os.environ.get("KAGGLE_USERNAME") or "YOURUSERNAME"
json.dump({"title": f"pipeline-survival-rung{RUNG}",
           "id": f"{KAGGLE_USER}/pipeline-survival-rung{RUNG}",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)

if not complete:
    print("\n" + "=" * 70)
    print(f"  PARTIAL: {len(df)}/{len(ev)} utterances measured. Nothing is lost.")
    print(f"    1. Output panel -> Create Dataset (or New Version) "
          f"-> pipeline-survival-rung{RUNG}")
    print( "    2. Attach that dataset back as an INPUT to this notebook")
    print( "    3. Re-commit. It skips what is already done.")
    print("=" * 70)
else:
    print(f"\nRUNG {RUNG}h / {EDIT_TYPE} COMPLETE -- publish as "
          f"pipeline-survival-rung{RUNG} and record the median in RUNLOG.md.")

print("\n" + json.dumps(PROV, indent=2))
df.head(10)

# ---------------------------------------------------------------------------
# CO-PRIMARY METRIC: attack success rate at a FIXED budget.
#
# Median eps50 is computed only over UNCENSORED utterances, so if the censoring
# rate differs between rungs -- and it will -- the medians describe different
# subpopulations and are not directly comparable. That is a censoring bias
# sitting directly under RQ1's headline.
#
# ASR@ceiling has no such problem: every utterance contributes, success or
# failure, so it is comparable across rungs by construction.
#
# The two measure different things and should both be reported:
#   ASR@ceiling -> can this model be STEERED to an exact target at all?
#   median eps50 -> when it can, how much perturbation does it take?
#
# Expect them to move in opposite directions. A better model is usually easier
# to aim (lower censoring) but harder to push (higher eps50). Reporting only
# one of them would tell half the story, and the flattering half depends on
# which one you pick.
# ---------------------------------------------------------------------------
ASR_AT_CEILING = float((~df.censored).mean())
print(f"\nASR@eps={EPS_HI:.0e}   : {ASR_AT_CEILING:.1%}  ({(~df.censored).sum()}/{len(df)})")
print(f"  <- censoring-free, uses all {len(df)} utterances, comparable across rungs")
print(f"median eps50 is over the {(~df.censored).sum()} steerable utterances only")

# ---------------------------------------------------------------------------
# LEFT-censoring: the mirror image of the block above, and the one this notebook
# had no check for. If eps50 lands at the bottom of the bracket, the attack
# succeeded at the smallest budget searched, so the value is an UPPER BOUND and
# the true eps50 is somewhere below it. Reporting it as a point estimate
# understates how cheap the attack is -- in the direction that flatters RQ2.
# ---------------------------------------------------------------------------
floor_hits = int((ok.eps50 <= EPS_LO * 1.15).sum()) if len(ok) else 0
PROV["left_censored"] = floor_hits
if floor_hits:
    print(f"\n*** LEFT-CENSORED: {floor_hits}/{len(ok)} eps50 values sit at the "
          f"bracket floor ({EPS_LO:.1e}) ***")
    print("    These are upper bounds, not measurements. Lower EPS_LO by a decade")
    print("    and re-run, or report them as censored-below and say so in the paper.")

if len(ok):
    b = [ok.eps50.sample(len(ok), replace=True).median() for _ in range(2000)]
    lo_ci, hi_ci = np.percentile(b, 2.5), np.percentile(b, 97.5)
    print(f"\neps50 bootstrap 95% CI spans {hi_ci/lo_ci:.2f}x")
    if hi_ci / lo_ci > 1.8:
        print("  WIDE. Between-rung differences must exceed this to be credible.")
        print("  If the ladder effect is smaller than the CI, raise CONFIRM_STEPS or")
        print("  run each condition twice -- but only if the effect turns out marginal.")

PROV["asr_at_ceiling"] = round(ASR_AT_CEILING, 4)
PROV["eps50_ci_ratio"] = round(float(hi_ci / lo_ci), 3) if len(ok) else None


---

## Record this in RUNLOG.md, then run the next combination

| Rung | T1 median eps50 | T1 censored | T5 median eps50 | T5 censored |
|---|---|---|---|---|
| 1 h | | | | |
| 5 h | | | | |
| 25 h | | | | |
| 100 h | | | | |

### What you are looking for

**RQ1:** eps50 should *rise* with training hours — more data, more perturbation needed. That is
the headline. It only counts after NB-02's iso-WER adjustment, since clean WER is a covariate.

**RQ2:** T1 should cost less than T5 at matched target length. Compare the
`eps50_per_token` column, not the raw one.

### Watch the censoring rate

If the 1 h rung shows **high censoring**, that is the paradox the plan predicted: a model whose
output is already unstable can be easy to *disrupt* yet hard to steer to a *specific* target.
That is a finding, not a failure. Report censoring alongside eps50 and use Kaplan–Meier rather
than silently dropping those rows.

### Next
**NB-06 — the LM-weight sweep.** Cheapest experiment in the thesis and the one that turns the
correlation into a mechanism. Prioritise it over everything except the ladder itself.

## 11 · RQ4 result

In [ ]:
# ============================================================================
#  RQ4 SUMMARY — survival of successful attacks through the deployment chain
# ============================================================================
EXPO = {1: 0.98, 5: 4.90, 25: 24.60, 100: 98.02}[RUNG]
d = pd.DataFrame(results)
verified = d[~d.censored]
n_ok = len(verified)

print(f"rung {RUNG}h   exposure {EXPO} h distinct audio")
print(f"raw attack success: {n_ok}/{len(d)} at eps {EPS_HI:.1e}")
if n_ok == 0:
    print("\nNo attack verified on the raw waveform, so survival is UNDEFINED for this rung.")
    print("Report the raw success rate and stop; do not report 0% survival -- that would")
    print("mean 'the chain broke the attack', and nothing was there to break.")
    SURV = pd.DataFrame()
else:
    rows = []
    for nm, kind, par in TRANSFORMS:
        if nm in SKIP:
            rows.append(dict(rung=RUNG, exposure_h=EXPO, transform=nm, kind=kind,
                             n=0, survives=None, broken=None, reverted=None,
                             survival_rate=None, note="encoder unavailable"))
            continue
        c = verified[f"surv_{nm}"].tolist()
        sv, bk, rv = c.count("survives"), c.count("broken"), c.count("reverted")
        rows.append(dict(rung=RUNG, exposure_h=EXPO, transform=nm, kind=kind,
                         n=len(c), survives=sv, broken=bk, reverted=rv,
                         survival_rate=sv / len(c) if c else None, note=""))
    SURV = pd.DataFrame(rows)
    SURV.to_csv(f"{WORK}/pipeline_survival.csv", index=False)
    print(f"\n{'transform':12s} {'n':>3s} {'surv':>5s} {'brok':>5s} {'revd':>5s} {'survival':>9s}")
    for r in SURV.itertuples():
        if r.survival_rate is None:
            print(f"{r.transform:12s}   --    --    --    --        SKIPPED"); continue
        print(f"{r.transform:12s} {r.n:3d} {r.survives:5d} {r.broken:5d} {r.reverted:5d} "
              f"{100*r.survival_rate:8.1f}%")

    _live = SURV[SURV.survival_rate.notna()]
    if len(_live):
        wk = _live.loc[_live.survival_rate.idxmin()]
        st = _live.loc[_live.survival_rate.idxmax()]
        # Bracket access, NOT attribute access: `transform` is also a pandas
        # Series method, so `wk.transform` returns the bound method, not the
        # column value. It printed "<bound method Series.transform of ...>" on
        # the first two runs. Same collision class as `flags` vs
        # DataFrame.flags, hit earlier in this project.
        print(f"\n  most destructive : {wk['transform']} -> "
              f"{100*wk['survival_rate']:.1f}% survive")
        print(f"  least destructive: {st['transform']} -> "
              f"{100*st['survival_rate']:.1f}% survive")
        _rev = _live.reverted.sum()
        print(f"  total 'reverted' outcomes: {_rev} of {_live.n.sum()} transform-utterances")
        if _rev:
            print("  'reverted' means the true transcript came back. A cheap transform that")
            print("  reverts attacks is a deployable defence -- the same argument RQ5 makes")
            print("  for the language model. Report those separately from 'broken'.")

PROV["rq4_exposure_h"] = EXPO
PROV["rq4_raw_success"] = int(n_ok)
PROV["rq4_transforms"] = [nm for nm, _, _ in TRANSFORMS if nm not in SKIP]
PROV["rq4_skipped"] = sorted(SKIP)
PROV["rq4_survival"] = ({r.transform: r.survival_rate for r in SURV.itertuples()}
                        if len(SURV) else {})
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), indent=2)

print("\n  NOT measured here, and the paper must keep saying so: over-the-air playback.")
print("  This is the digital chain only. No microphone, no room, no loudspeaker.")
print(f"\n  Publish as pipeline-survival-rung{RUNG}. Run the other three rungs, then read")
print("  survival against exposure -- that cross-tab is the RQ4 result, not any single rung.")
